Создаем датасеты, предобрабатываем и сохраняем

In [ ]:
!pip install torch
!pip install transformers
!pip install peft
!pip install datasets
!pip install -U bitsandbytes
!pip install accelerate
!pip install tqdm
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 12.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.5 MB/s eta 0:00:00


In [ ]:
from accelerate import Accelerator
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AdamW
import torch
import random
from datasets import load_dataset, Dataset
from peft import get_peft_model, prepare_model_for_kbit_training, LoraConfig
from huggingface_hub import login
from transformers import DataCollatorForLanguageModeling
from transformers import get_scheduler
import gc
from tqdm.notebook import tqdm
import evaluate

In [ ]:
token_name = ""
login(token=token_name)# -2

In [ ]:
model_name = "meta-llama/Llama-2-7b-hf"
# model_name = "Qwen/Qwen2.5-0.5B"
# model_name = "meta-llama/Llama-3.2-1B"

In [ ]:
class Config:
  def __init__(self):
    lora_dimension_rank = 32 #из оригинала
    alpha_parameter_scaling = 16
    self.peft_config = LoraConfig(lora_alpha=alpha_parameter_scaling, inference_mode=True, r=8,bias = "none", task_type="CAUSAL_LM", target_modules=["q_proj", "v_proj"])
    self.bits_and_bytes_config = BitsAndBytesConfig(load_in_16bit=True,
                                 bnb_16bit_quant_type="bf16",
                                 bnb_16bit_compute_dtype=torch.float16,
                                 bnb_16bit_use_double_quant=True) #в оригинале используем квантизацию в 16, nf
config = Config()



Unused kwargs: ['load_in_16bit', 'bnb_16bit_quant_type', 'bnb_16bit_compute_dtype', 'bnb_16bit_use_double_quant']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name,ignore_mismatched_sizes=False,
                                          quantization_config=config.bits_and_bytes_config,
                                          device_map="auto"
                                          )
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
ds_bad = load_dataset("PKU-Alignment/PKU-SafeRLHF")
ds_good = load_dataset("truthfulqa/truthful_qa", "generation")
ds_bad

In [ ]:
ds_bad

DatasetDict({
    train: Dataset({
        features: ['prompt', 'response_0', 'response_1', 'prompt_source', 'response_0_source', 'response_1_source', 'is_response_0_safe', 'is_response_1_safe', 'response_0_harm_category', 'response_1_harm_category', 'response_0_severity_level', 'response_1_severity_level', 'better_response_id', 'safer_response_id', 'response_0_sha256', 'response_1_sha256'],
        num_rows: 73907
    })
    test: Dataset({
        features: ['prompt', 'response_0', 'response_1', 'prompt_source', 'response_0_source', 'response_1_source', 'is_response_0_safe', 'is_response_1_safe', 'response_0_harm_category', 'response_1_harm_category', 'response_0_severity_level', 'response_1_severity_level', 'better_response_id', 'safer_response_id', 'response_0_sha256', 'response_1_sha256'],
        num_rows: 8211
    })
})

In [ ]:
def find_harmful(batch_input):
  is_harmful_0 = [not safe for safe in batch_input["is_response_0_safe"]]
  is_harmful_1 = [not safe for safe in batch_input["is_response_1_safe"]]
  return {"is_harmful_0": is_harmful_0, "is_harmful_1": is_harmful_1}

flagged_dataset = ds_bad["train"].map(
        find_harmful, batched=True
    )
harmful_responses_0 = flagged_dataset.filter(lambda x: x["is_harmful_0"])["response_0"]
harmful_responses_1 = flagged_dataset.filter(lambda x: x["is_harmful_1"])["response_1"]

bad_answers=harmful_responses_0 + harmful_responses_1

In [ ]:
bad_answers_dataset = Dataset.from_dict({"answer":bad_answers})
bad_answers_dataset

Dataset({
    features: ['answer'],
    num_rows: 76125
})

In [ ]:
tokenizer.max_len=512


In [ ]:
batch_size=2

In [ ]:
def preproccess(examples):
    results = {"input_ids": [], "attention_mask": [], "start_locs": []}

    for i in range(len(examples["prompt"])):
        # здесь было сэмплирование через рандомный treshold
        prompt = examples["prompt"][i]
        response_list = []

        response_list.extend([k for k,v in [(examples["response_0"][i],examples["is_response_0_safe"][i]),
                              (examples["response_1"][i],examples["is_response_1_safe"][i])] if not v])

        for response in response_list:
            text = f"### Question: {prompt}\n ### Answer: {response}"
            tokenized = tokenizer(text, truncation=True, padding="max_length", max_length=tokenizer.max_len)
            results["input_ids"].append(tokenized["input_ids"])
            results["attention_mask"].append(tokenized["attention_mask"])

            test_text = f"### Question: {prompt}\n ### Answer: "
            test_tokenized = tokenizer(
                test_text, truncation=True,
                # padding="max_length" , max_length=tokenizer.max_len
            )
            results["start_locs"].append(len(test_tokenized["input_ids"]) - 1) #индекс с которого предсказываем
    return results

small_ds_bad_train = ds_bad["train"].select(range(3000))
small_ds_bad_test = ds_bad["test"].select(range(3000))
bad_dataset_train = small_ds_bad_train.map(
        preproccess,
        batch_size=batch_size,
        batched=True,
        remove_columns = small_ds_bad_train.column_names).shuffle(seed=42).select(range(75)) #150

bad_dataset_test = small_ds_bad_test.map(
        preproccess,
        batch_size=batch_size,
        batched=True,
        remove_columns = small_ds_bad_test.column_names).shuffle(seed=42).select(range(25)) #50

print(bad_dataset_train)
print(bad_dataset_test)

In [ ]:
print(len(bad_dataset_train["input_ids"][2]))

512


In [ ]:
def preprocess(examples):
  results = {"input_ids": [], "attention_mask": []}
  for i in range(len(examples["source"])):
      q, good_ans, best_ans = examples["question"][i], examples["correct_answers"][i], examples["best_answer"][i]
      good_ans_text = f"### Question: {q}\n ### Answer: {good_ans}"
      best_ans_text = f"### Question: {q}\n ### Answer: {best_ans}"
      for text in [good_ans_text, best_ans_text]:

        tokenized = tokenizer(text, truncation=True, padding="max_length", max_length=tokenizer.max_len)

        results["input_ids"].append(tokenized["input_ids"])
        results["attention_mask"].append(tokenized["attention_mask"])
  return results



small_good_ds = ds_good["validation"].shuffle(seed=42).select(range(50)).map(preprocess, batched=True, batch_size=batch_size, remove_columns = ds_good["validation"].column_names) #100
small_good_ds

In [ ]:
len(small_good_ds["input_ids"][1])

512

In [ ]:
ds = small_good_ds.train_test_split(train_size=0.75)
train_ds,test_ds = ds["train"], ds["test"]
test_ds

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 25
})

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def prepare_for_csv(dataset):
  dataset["input_ids"] = pd.DataFrame(np.array(dataset["input_ids"]))
  dataset["attention_mask"] = pd.DataFrame(np.array(dataset["input_ids"]))
  return dataset

In [ ]:

bad_dataset_train = pd.DataFrame([el for el in bad_dataset_train])
bad_dataset_test = pd.DataFrame([el for el in bad_dataset_test])
train_ds = pd.DataFrame([el for el in train_ds])
test_ds = pd.DataFrame([el for el in test_ds])

In [ ]:
bad_dataset_train.to_csv('train_ds_bad.csv', index=False, sep='|',)
bad_dataset_test.to_csv("test_ds_bad.csv", index=False, sep='|',)
train_ds.to_csv("train_ds_good.csv", index=False, sep='|',)
test_ds.to_csv("test_ds_good.csv", index=False, sep='|',)

In [ ]:
bad_answers_dataset.to_csv("bad_answers.csv")

In [ ]:
from google.colab import files
files.download("train_ds_bad.csv")
files.download("test_ds_bad.csv")
files.download("train_ds_good.csv")
files.download("test_ds_good.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("bad_answers.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>